In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook


output_notebook()
hv.extension('bokeh')

font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

In [ ]:
monkey = 'fiona' # 'yasmin'  or 'fiona' 
base_path = Path.cwd().parent / 'data' / 'csst_trials_pkls'
# filepath = base_path / f'all_{monkey}_CSST_trials_df.pkl'
filepath = base_path / f'all_{monkey}_CSST_trials_df_including_all_200_neuron_fields.pkl'
# filepath = base_path / f'no_filters_{monkey}_CSST_trials_df_including_all_200_neuron_fields.pkl'

df = pd.read_pickle(filepath)

print(df.info())
# df.iloc[:2]
df.head()

In [ ]:
df['filename'].apply(lambda x: x.split('.')[0][-1]).unique()

In [ ]:
neurons_list = []

def extract_neuron_ids(row, neurons_list):
    trial_neuron_ids = [f"{row['trial_session']}_{key}" for key in row['neural_data'].keys()]
    neurons_list += trial_neuron_ids
    return trial_neuron_ids

df.apply(
    lambda row: extract_neuron_ids(row, neurons_list), 
    axis=1
)

neurons_set = set(neurons_list)
print(f'Total unique neurons across all sessions: {len(neurons_set)}') 

In [ ]:
# Drop unnecessary columns
cols_to_drop = [
    'vPos', 'hPos', 'vVel', 'hVel', 'speed',
    'set', 'direction'
]
df.drop(columns=cols_to_drop, inplace=True)
print(f"DataFrame shape after dropping columns: {df.shape}")
df.head()

In [ ]:
# reorder columns
new_order = [
    'filename', 'trial_name', 'reaction_time', 
    'go_cue', 'stop_cue', 'trial_failed', 
    'first_relevant_saccade', 'segs_durations', 'segs_times',
    'trial_length', 'ssd_len', 'ssd_number',
    'screen_rotation', 'neural_data', 'saccades', 
    'blinks', 'dir', 'flags',
    'type', 'trial_session', 'trial_number',
]

df = df[new_order]
df.head()

In [ ]:
# load monkey's cell db from xlsx file
cell_db_path = Path.cwd().parent / 'data' / f'database_sst'
cell_db = pd.read_excel(cell_db_path / f'SST_{monkey}_cells_db.xlsx')
print(cell_db.info())
cell_db.head()

In [ ]:
cell_db[
    (cell_db['cell_type'] == 'msn') & 
    (cell_db['grade'] <= 8)
].shape

In [ ]:
df.iloc[0].neural_data.keys()

In [ ]:
if monkey == 'fiona':
    cell_db.at[429, 'fe_after_stability'] = 1017

In [ ]:
cell_db.columns

In [ ]:
cell_db[
    cell_db['fe_after_stability'].apply(
        lambda row: isinstance(row, str)
)].apply(
    lambda row: [
        np.fromstring(
            row[key][1:-1], sep=' ', dtype=np.int16
        )
        for key in ['fb_after_stablility', 'fe_after_stability']
    ],
    axis=1, result_type='expand'
)

# np.fromstring(tmp[1:-1], sep=' ', dtype=np.int16)

In [ ]:
def get_row_stable_trials_total(row):
    if isinstance(row['fe_after_stability'], str):
        fb_stable = np.fromstring(
            row['fb_after_stablility'][1:-1], sep=' ', dtype=np.int16
        )
        fe_stable = np.fromstring(
            row['fe_after_stability'][1:-1], sep=' ', dtype=np.int16
        )
        return fe_stable.sum() - fb_stable.sum()
    elif isinstance(row['fe_after_stability'], int):
        return row['fe_after_stability'] - row['fb_after_stablility']
    else:
        raise ValueError("Unexpected data type in 'fe_after_stability' column")
    
row = cell_db.iloc[0]
row = cell_db[
    cell_db['fe_after_stability'].apply(
        lambda row: isinstance(row, str)
)].iloc[0]
get_row_stable_trials_total(row)
# cell_db[(cell_db.apply(get_row_stable_trials_total, axis=1) < 0)]
stable_trials = cell_db[cell_db['cell_type'].isin(['msn', 'pu msn'])].apply(get_row_stable_trials_total, axis=1)
stable_trials.sum()

In [ ]:
# Function to extract session and trial number from filename
def parse_filename(filename):
    """
    Parse filename like 'fi210824a.0614' into components
    Returns: (session, plexon_session, trial_number)
    """
    parts = filename.split('.')
    if len(parts) != 2:
        return None, None, None
    
    prefix = parts[0]  # 'fi210824a'
    trial_num = parts[1]  # '0614'
    
    if len(prefix) < 9:  # minimum: 'fi' + 6 digits + 'a'
        return None, None, None
    
    session = prefix[:-1]  # 'fi210824' (remove plexon session letter)
    plexon_session = prefix[-1]  # 'a'
    
    return session, plexon_session, int(trial_num)

# Test the function
test_filename = df.iloc[0]['filename']
print(f"Test filename: {test_filename}")
session, plexon_session, trial_num = parse_filename(test_filename)
print(f"Parsed: session='{session}', plexon_session='{plexon_session}', trial_num={trial_num}")

# Check a few more examples
print("\nTesting with more filenames:")
for i in range(5):
    fname = df.iloc[i]['filename']
    session, plexon_session, trial_num = parse_filename(fname)
    print(f"{fname} -> session='{session}', plexon_session='{plexon_session}', trial_num={trial_num}")

In [ ]:
# Check the column names for stability columns
print("Cell DB columns:")
print(cell_db.columns.tolist())

print(f"\nSample stability columns:")
stability_cols = ['fb_after_stablility', 'fe_after_stability']
for col in stability_cols:
    if col in cell_db.columns:
        print(f"{col}: {cell_db[col].head().tolist()}")
        print(f"  Data types: {cell_db[col].apply(type).value_counts()}")
    else:
        print(f"❌ Column '{col}' not found!")

# Check neural data keys format
print(f"\nSample neural data keys:")
sample_neural_data = df.iloc[0]['neural_data']
print(f"Type: {type(sample_neural_data)}")
if isinstance(sample_neural_data, dict):
    print(f"Keys (first 10): {list(sample_neural_data.keys())[:10]}")
    print(f"Key types: {[type(k) for k in list(sample_neural_data.keys())[:5]]}")
else:
    print(f"Not a dict: {sample_neural_data}")

In [ ]:
# Helper functions for parallel processing
def is_trial_in_stability_range(fb_stability, fe_stability, trial_num):
    """Check if trial is within ANY stability range, handling multiple ranges properly"""
    # Wrap arguments in lists and create numpy arrays, then flatten
    try:
        fb_array = np.array([int(fb_stability)])
        fe_array = np.array([int(fe_stability)])
    except ValueError as e:
        if str(e).startswith("invalid literal for int() with base 10: "):
            fb_array = np.fromstring(fb_stability[1:-1], sep=' ', dtype=np.int16)
            fe_array = np.fromstring(fe_stability[1:-1], sep=' ', dtype=np.int16)
        else:
            raise e
    
    # Get minimum length to ensure we don't go out of bounds
    min_len = min(len(fb_array), len(fe_array))
    
    # Check each stability range pair
    for i in range(min_len):
        fb_start = fb_array[i]
        fe_end = fe_array[i]
        
        # Skip invalid values
        if pd.isna(fb_start) or pd.isna(fe_end):
            continue
            
        try:
            # Check if trial falls within this range
            if int(fb_start) <= trial_num <= int(fe_end):
                return True  # Trial is within this stability range
        except (ValueError, TypeError):
            continue  # Skip this range if conversion fails
    
    return False  # Trial is not within any stability range

def parse_filename(filename):
    """Parse filename like 'fi210824a.0614' into components"""
    parts = filename.split('.')
    if len(parts) != 2:
        return None, None, None
    
    prefix = parts[0]  # 'fi210824a'
    trial_num = parts[1]  # '0614'
    
    if len(prefix) < 9:  # minimum: 'fi' + 6 digits + 'a'
        return None, None, None
    
    session = prefix[:-1]  # 'fi210824' (remove plexon session letter)
    plexon_session = prefix[-1]  # 'a'
    
    return session, plexon_session, int(trial_num)

def process_trial_chunk(args):
    """Process a chunk of trials - this function will run in parallel"""
    trial_chunk, cells_df_dict = args
    
    # Convert cells_df_dict back to DataFrame
    cells_df = pd.DataFrame(cells_df_dict)
    
    unified_data = []
    
    for _, trial_row in trial_chunk.iterrows():
        # Parse filename to get session info
        filename = trial_row['filename']
        session, plexon_session, trial_num = parse_filename(filename)
        
        if session is None:
            continue
        
        # Get neural data for this trial
        # neural_data = trial_row.get('neural_data', {})
        neural_data = trial_row['neural_data']
        if not isinstance(neural_data, dict):
            neural_data = {}
            raise ValueError(f"Unexpected neural_data format in trial '{filename}'")
        
        # Find cells that match session and plexon session
        session_cells = cells_df[
            (cells_df['session'] == session) #& 
            # (cells_df['plexon_session'] == plexon_session) !!!!!!!!! This could be the issue
        ]
        
        # Filter by stability range
        stable_cells = []
        for _, cell_row in session_cells.iterrows():
            if is_trial_in_stability_range(
                cell_row['fb_after_stablility'], 
                cell_row['fe_after_stability'], 
                trial_num
            ):
                stable_cells.append(cell_row)
        
        # Create one row per cell for this trial
        for cell_row in stable_cells:
            maestro_id_matlab = cell_row['maestro_ID']  # MATLAB 1-based ID
            
            # Fix MATLAB/Python indexing: MATLAB uses 1-based, Python uses 0-based
            # Neural data keys are 0-based (Python), maestro_ID in xlsx is 1-based (MATLAB)
            maestro_id_python = maestro_id_matlab - 1  # Convert to Python 0-based
            cell_neural_data = neural_data.get(maestro_id_python, [])
            
            # Create unified row
            unified_row = {
                # Cell information
                'cell_ID': cell_row['cell_ID'],
                'cell_type': cell_row['cell_type'], 
                'maestro_ID': maestro_id_matlab,  # Keep original MATLAB ID for reference
                'problem': cell_row['problem'],
                'grade': cell_row['grade'],
                
                # Core trial information
                'filename': trial_row['filename'],
                'trial_name': trial_row['trial_name'],
                'reaction_time': trial_row['reaction_time'],
                'go_cue': trial_row['go_cue'],
                'stop_cue': trial_row['stop_cue'], 
                'trial_failed': trial_row['trial_failed'],
                'ssd_len': trial_row['ssd_len'],
                'ssd_number': trial_row['ssd_number'],
                'type': trial_row['type'],
                
                # Additional trial information
                'first_relevant_saccade': trial_row['first_relevant_saccade'],
                'segs_durations': trial_row['segs_durations'],
                'segs_times': trial_row['segs_times'],
                'trial_length': trial_row['trial_length'],
                'screen_rotation': trial_row['screen_rotation'],
                'saccades': trial_row['saccades'],
                'blinks': trial_row['blinks'],
                'dir': trial_row['dir'],
                
                # Neural data for this specific cell
                'neural_data': cell_neural_data,
                
                # Additional useful columns
                'session': session,
                'plexon_session': plexon_session,
                'trial_number': trial_num,
                'trial_session': trial_row['trial_session']
            }
            
            unified_data.append(unified_row)
    
    return unified_data

# PARALLEL VERSION - Create the CORRECTED unified DataFrame with ProcessPoolExecutor
def create_unified_dataframe_parallel(trials_df, cells_df, chunk_size=100, max_workers=None):
    """
    Create a unified DataFrame with one row per cell-trial combination using parallel processing.
    
    Parameters:
    -----------
    trials_df : pd.DataFrame
        DataFrame with trial data
    cells_df : pd.DataFrame  
        DataFrame with cell information
    chunk_size : int
        Number of trials to process in each chunk
    max_workers : int
        Number of parallel workers (None = use CPU count)
    
    Returns:
    --------
    pd.DataFrame : Unified DataFrame with cell-trial combinations
    """
    print("Creating unified DataFrame with corrections (PARALLEL VERSION)...")
    print(f"Processing {len(trials_df)} trials and {len(cells_df)} cells...")
    
    # Convert cells_df to dict for pickling (required for ProcessPoolExecutor)
    cells_df_dict = cells_df.to_dict('records')
    
    # Split trials into chunks
    trial_chunks = []
    for i in range(0, len(trials_df), chunk_size):
        chunk = trials_df.iloc[i:i+chunk_size]
        trial_chunks.append((chunk, cells_df_dict))
    
    print(f"Split into {len(trial_chunks)} chunks of ~{chunk_size} trials each")
    
    # Determine number of workers
    if max_workers is None:
        max_workers = min(mp.cpu_count(), len(trial_chunks))
    
    print(f"Using {max_workers} parallel workers")
    
    # Process chunks in parallel
    all_unified_data = []
    all_unstable_trials = []    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {executor.submit(process_trial_chunk, args): i 
                  for i, args in enumerate(trial_chunks)}
        
        # Collect results with progress bar
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing chunks"):
            chunk_results = future.result()
            all_unified_data.extend(chunk_results)
    
    # Create final DataFrame
    unified_df = pd.DataFrame(all_unified_data)

    print(f"\nUnified DataFrame created with {len(unified_df)} rows")
    print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
    print(f"Unique trials: {unified_df['filename'].nunique()}")
    
    return unified_df

# Original sequential version (for comparison or fallback)
def create_unified_dataframe_corrected(trials_df, cells_df):
    """Sequential version - kept for fallback or comparison"""
    unified_data = []
    
    print("Creating unified DataFrame with corrections (SEQUENTIAL VERSION)...")
    print(f"Processing {len(trials_df)} trials and {len(cells_df)} cells...")
    
    for trial_idx, trial_row in tqdm(trials_df.iterrows(), total=len(trials_df), desc="Processing trials"):
        # Parse filename to get session info
        filename = trial_row['filename']
        session, plexon_session, trial_num = parse_filename(filename)
        
        if session is None:
            # raise ValueError(f"Could not parse filename: {filename}")
            continue
        
        # Get neural data for this trial
        # neural_data = trial_row.get('neural_data', {})
        neural_data = trial_row['neural_data']
        if not isinstance(neural_data, dict):
            # neural_data = {}
            raise ValueError(f"Unexpected neural_data format in trial '{filename}'")
        
        # Find cells that match session and plexon session
        session_cells = cells_df[
            (cells_df['session'] == session) #& 
            # (cells_df['plexon_session'] == plexon_session) !!!!!!!!!! This could be the issue
        ]
        
        # Filter by stability range
        stable_cells = []
        for _, cell_row in session_cells.iterrows():
            if is_trial_in_stability_range(
                cell_row['fb_after_stablility'], 
                cell_row['fe_after_stability'], 
                trial_num
            ):
                stable_cells.append(cell_row)
            else:
                error = f"Trial {trial_num} not in stability range for cell {cell_row['cell_ID']} ins session {session}"
                raise ValueError(error)
        
        # Create one row per cell for this trial
        for cell_row in stable_cells:
            maestro_id_matlab = cell_row['maestro_ID']  # MATLAB 1-based ID
            
            # Fix MATLAB/Python indexing: MATLAB uses 1-based, Python uses 0-based
            # Neural data keys are 0-based (Python), maestro_ID in xlsx is 1-based (MATLAB)
            maestro_id_python = maestro_id_matlab - 1  # Convert to Python 0-based
            cell_neural_data = neural_data.get(maestro_id_python, [])
            
            # Create unified row
            unified_row = {
                # Cell information
                'cell_ID': cell_row['cell_ID'],
                'cell_type': cell_row['cell_type'], 
                'maestro_ID': maestro_id_matlab,  # Keep original MATLAB ID for reference
                'problem': cell_row['problem'],
                
                # Core trial information
                'filename': trial_row['filename'],
                'trial_name': trial_row['trial_name'],
                'reaction_time': trial_row['reaction_time'],
                'go_cue': trial_row['go_cue'],
                'stop_cue': trial_row['stop_cue'], 
                'trial_failed': trial_row['trial_failed'],
                'ssd_len': trial_row['ssd_len'],
                'ssd_number': trial_row['ssd_number'],
                'type': trial_row['type'],
                
                # Additional trial information
                'first_relevant_saccade': trial_row['first_relevant_saccade'],
                'segs_durations': trial_row['segs_durations'],
                'segs_times': trial_row['segs_times'],
                'trial_length': trial_row['trial_length'],
                'screen_rotation': trial_row['screen_rotation'],
                'saccades': trial_row['saccades'],
                'blinks': trial_row['blinks'],
                'dir': trial_row['dir'],
                
                # Neural data for this specific cell
                'neural_data': cell_neural_data,
                
                # Additional useful columns
                'session': session,
                'plexon_session': plexon_session,
                'trial_number': trial_num,
                'trial_session': trial_row['trial_session']
            }
            
            unified_data.append(unified_row)
    
    unified_df = pd.DataFrame(unified_data)
    print(f"\nUnified DataFrame created with {len(unified_df)} rows")
    print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
    print(f"Unique trials: {unified_df['filename'].nunique()}")
    
    return unified_df

# Test with sample data first - using parallel version
sample_df = df.sample(10, random_state=42)
print("Testing parallel version with 10 sample trials...")
unified_df_corrected = create_unified_dataframe_parallel(
    sample_df, 
    cell_db, 
    chunk_size=5, 
    max_workers=2
)
unified_df_corrected

In [ ]:
unified_df_corrected['filename'].nunique(), sample_df['filename'].nunique()

In [ ]:
# sample_df

In [ ]:
# Use the PARALLEL version for much faster processing
print("Using the PARALLEL unified DataFrame function...")
print("This function properly handles:")
print("1. MATLAB/Python indexing conversion")
print("2. Stability range filtering") 
print("3. Multiple stability ranges")
print("4. Parallel processing for speed")

# Run with full dataset - using optimal chunk size and max workers
print(f"\nProcessing full dataset with {len(df)} trials...")
unified_df = create_unified_dataframe_parallel(
    df, 
    cell_db, 
    chunk_size=200,  # Adjust based on memory vs speed tradeoff
    max_workers=None  # Use all available CPU cores
)

In [ ]:
# Analyze the unified DataFrame with additional columns
print("=== UNIFIED DATAFRAME ANALYSIS (WITH ADDITIONAL COLUMNS) ===")
print(f"Shape: {unified_df.shape}")
print(f"Memory usage: {unified_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nColumn info:")
print(f"Total columns: {len(unified_df.columns)}")
print(f"Columns: {list(unified_df.columns)}")

print(f"\nData summary:")
print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
print(f"Unique trials: {unified_df['filename'].nunique()}")  
print(f"Unique sessions: {unified_df['session'].nunique()}")

print(f"\nCell types distribution:")
print(unified_df['cell_type'].value_counts())

print(f"\nTrial types in unified data:")
if 'type' in unified_df.columns:
    print(unified_df['type'].value_counts())

print(f"\nTrial name samples:")
trial_name_samples = unified_df['trial_name'].value_counts()
print(trial_name_samples.head(10))

print(f"\nNeural data statistics:")
neural_data_lengths = unified_df['neural_data'].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)
print(f"Mean spikes per trial: {neural_data_lengths.mean():.1f}")
print(f"Max spikes per trial: {neural_data_lengths.max()}")
print(f"Trials with no spikes: {(neural_data_lengths == 0).sum()}")

# Show sample rows with new columns
print(f"\nSample data with key columns:")
display_cols = ['cell_ID', 'cell_type', 'maestro_ID', 'filename', 'trial_name', 'type', 'reaction_time', 
                'first_relevant_saccade', 'trial_length', 'dir', 'problem']
available_cols = [col for col in display_cols if col in unified_df.columns]
print(f"Available columns: {available_cols}")
print(unified_df[available_cols].head(3))

# Check additional column data types and samples
print(f"\nAdditional column samples:")
additional_cols = ['segs_durations', 'segs_times', 'screen_rotation', 'saccades', 'blinks']
for col in additional_cols:
    if col in unified_df.columns:
        sample_val = unified_df[col].iloc[0]
        print(f"{col}: {type(sample_val).__name__} - {str(sample_val)[:100]}{'...' if len(str(sample_val)) > 100 else ''}")

In [ ]:
print(unified_df[['cell_type', 'cell_ID']].drop_duplicates('cell_ID')['cell_type'].value_counts())

unified_df['grade'].value_counts()


In [ ]:
# Get MSN cell IDs from cell_db
msn_cells_in_db = set(cell_db[cell_db['cell_type'] == 'msn']['cell_ID'].unique())

# Get MSN cell IDs from unified_df
msn_cells_in_unified = set(unified_df[unified_df['cell_type'] == 'msn']['cell_ID'].unique())

# Find MSN cells in cell_db but not in unified_df
msn_missing = msn_cells_in_db - msn_cells_in_unified

print(f"MSN cells in cell_db: {len(msn_cells_in_db)}")
print(f"MSN cells in unified_df: {len(msn_cells_in_unified)}")
print(f"MSN cells missing from unified_df: {len(msn_missing)}")
# the missing cells MSN cells are in non CSST trials or have no stable trials

In [ ]:
unified_df[unified_df['cell_type'].isin(['msn']) & (unified_df['grade'] <= 8)]['cell_ID'].nunique()

In [ ]:
# Save the unified DataFrame
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
save_path.mkdir(exist_ok=True)

# Save as pickle for efficient loading
pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'
# unified_df.to_pickle(pickle_file)
print(f"Unified DataFrame saved to: {pickle_file}")

# # Also save a CSV version for easy inspection (but this will be larger)
# csv_file = save_path / f'unified_{monkey}_cell_trial_data.csv'
# # For CSV, convert neural_data to string representation to avoid issues
# csv_df = unified_df.copy()
# csv_df['neural_data'] = csv_df['neural_data'].apply(lambda x: str(x) if x else "[]")
# csv_df.to_csv(csv_file, index=False)
# print(f"CSV version saved to: {csv_file}")

# print(f"\nFile sizes:")
# print(f"Pickle: {pickle_file.stat().st_size / 1e6:.1f} MB") 
# print(f"CSV: {csv_file.stat().st_size / 1e6:.1f} MB")

# print(f"\nDataFrame ready for neural analysis!")
# print(f"Use: pd.read_pickle('{pickle_file}') to load the unified data")